<a href="https://colab.research.google.com/github/Jasp3r16/thesis_generative_timber/blob/main/main_geometry_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-12
### Properties script
**Goal:** Main script that generates the geometry dataset that acts as input for the structural analysis in grasshopper, this script can take several variables as input at generates a CSV file made up of properties that can generate this digital geometry to a CAD geometry   
**Inputs:**   
*   Define base mesh
*   Allowed movement for different vertices
*   divisions over allowed movement

**Outputs:**

*   CSV file, with sample id, vertices, coordinates and edges

## Mounting Google drive

In [1]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Mounted at /content/drive
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


## Top layer vertices

In [ ]:
# @title
import pandas as pd
import numpy as np
import random

# --- CONFIGURATIE ---
GRID_SIZE_X = 4  # Aantal cellen in X
GRID_SIZE_Y = 4  # Aantal cellen in Y
EDGE_LENGTH = 100.0
DIVISIONS = 8    # Aantal divisies (bijv. 8)
NUM_SAMPLES = 5  # Aantal samples om te genereren voor deze test

def get_valid_shifts(divisions, edge_length):
    """
    Berekent de toegestane verschuivingen volgens de 'remove first and last' regel.
    Bereik is [-0.5 * L, 0.5 * L].
    Bij 8 divisies zijn de stappen: -4, -3, -2, -1, 0, 1, 2, 3, 4 (in achtsten).
    We verwijderen -4 en 4.
    Blijft over: -3 tot 3.
    """
    half_div = divisions // 2
    # Genereer alle stappen van -half tot +half
    all_steps = list(range(-half_div, half_div + 1))

    # Verwijder de eerste en laatste (de extremen die overlap veroorzaken)
    valid_steps = all_steps[1:-1]

    # Converteer steps naar daadwerkelijke afstanden
    # Shift = (step / divisions) * edge_length
    # Bijv: stap 1 bij 8 divisies en lengte 100 = (1/8)*100 = 12.5
    valid_shifts = [(step / divisions) * edge_length for step in valid_steps]

    return valid_shifts

def generate_top_layer_dataset(num_samples):
    valid_shifts = get_valid_shifts(DIVISIONS, EDGE_LENGTH)

    vertex_data = []

    num_nodes_x = GRID_SIZE_X + 1
    num_nodes_y = GRID_SIZE_Y + 1

    for sample_id in range(num_samples):
        vertex_idx = 0
        for i in range(num_nodes_y):      # Loop rijen (y)
            for j in range(num_nodes_x):  # Loop kolommen (x)

                # Basis positie
                base_x = j * EDGE_LENGTH
                base_y = i * EDGE_LENGTH
                base_z = 0.0

                shift_x = 0.0
                shift_y = 0.0
                v_type = "Inner"

                # Bepaal Type en Constraints
                is_left_edge = (j == 0)
                is_right_edge = (j == num_nodes_x - 1)
                is_bottom_edge = (i == 0)
                is_top_edge = (i == num_nodes_y - 1)

                is_corner = (is_left_edge or is_right_edge) and (is_bottom_edge or is_top_edge)
                is_edge = (is_left_edge or is_right_edge or is_bottom_edge or is_top_edge) and not is_corner

                if is_corner:
                    v_type = "Corner"
                    # Corners bewegen NIET (Source 42)
                    pass

                elif is_edge:
                    v_type = "Edge"
                    # Edge vertices bewegen langs hun rand om de bounding box te behouden (Source 41)
                    if is_left_edge or is_right_edge:
                        # Verticale rand: mag alleen in Y bewegen
                        shift_y = random.choice(valid_shifts)
                    else: # Bottom of Top edge
                        # Horizontale rand: mag alleen in X bewegen
                        shift_x = random.choice(valid_shifts)

                else:
                    v_type = "Inner"
                    # Inner vertices bewegen in X en Y
                    shift_x = random.choice(valid_shifts)
                    shift_y = random.choice(valid_shifts)

                # Bereken definitieve positie
                final_x = base_x + shift_x
                final_y = base_y + shift_y
                final_z = base_z

                vertex_data.append({
                    "sample_id": sample_id,
                    "vertex_index": vertex_idx,
                    "x": round(final_x, 3),
                    "y": round(final_y, 3),
                    "z": round(final_z, 3),
                    # "type": v_type # Optioneel voor controle
                })

                vertex_idx += 1

    return pd.DataFrame(vertex_data)

def generate_static_edges():
    """Genereert de edge list (connectiviteit) voor een grid."""
    edges = []
    edge_id = 0
    num_nodes_x = GRID_SIZE_X + 1
    num_nodes_y = GRID_SIZE_Y + 1

    # Horizontale edges
    for i in range(num_nodes_y):
        for j in range(GRID_SIZE_X):
            v1 = i * num_nodes_x + j
            v2 = v1 + 1
            edges.append({"edge_id": edge_id, "u": v1, "v": v2})
            edge_id += 1

    # Verticale edges
    for i in range(GRID_SIZE_Y):
        for j in range(num_nodes_x):
            v1 = i * num_nodes_x + j
            v2 = v1 + num_nodes_x
            edges.append({"edge_id": edge_id, "u": v1, "v": v2})
            edge_id += 1

    return pd.DataFrame(edges)

# --- UITVOEREN ---
df_vertices = generate_top_layer_dataset(NUM_SAMPLES)
df_edges = generate_static_edges()

# Opslaan
df_vertices.to_csv("dataset_vertices.csv", index=False)
df_edges.to_csv("static_edges.csv", index=False)

print("Vertices Sample (eerste 10 regels):")
print(df_vertices.head(10))
print("\nEdges Sample (eerste 5 regels):")
print(df_edges.head(5))

Vertices Sample (eerste 10 regels):
   sample_id  vertex_index      x      y    z
0          0             0    0.0    0.0  0.0
1          0             1   62.5    0.0  0.0
2          0             2  200.0    0.0  0.0
3          0             3  275.0    0.0  0.0
4          0             4  400.0    0.0  0.0
5          0             5    0.0  137.5  0.0
6          0             6   75.0   75.0  0.0
7          0             7  187.5   62.5  0.0
8          0             8  262.5  112.5  0.0
9          0             9  400.0   75.0  0.0

Edges Sample (eerste 5 regels):
   edge_id  u  v
0        0  0  1
1        1  1  2
2        2  2  3
3        3  3  4
4        4  5  6


## Bottom layer vertices + Top layer vertices

In [ ]:
# @title
import pandas as pd
import numpy as np
import random

# --- CONFIGURATIE ---
GRID_CELLS_X = 4       # Aantal cellen in X (dus 5 vertices breed)
GRID_CELLS_Y = 4       # Aantal cellen in Y
EDGE_LENGTH = 100.0    # Afmeting van een cel
LAYER_HEIGHT = 100.0   # Afstand tussen top en bottom layer
DIVISIONS = 8          # Aantal stappen voor de discrete verplaatsing
NUM_SAMPLES = 5        # Aantal samples om te genereren

def get_valid_shifts(divisions, edge_length):
    """Berekent de toegestane verschuivingen (verwijdert extremen)."""
    half_div = divisions // 2
    all_steps = list(range(-half_div, half_div + 1))
    valid_steps = all_steps[1:-1] # Verwijder eerste en laatste
    valid_shifts = [(step / divisions) * edge_length for step in valid_steps]
    return valid_shifts

def bilinear_interpolate(p00, p10, p01, p11, u, v):
    """
    Interpoleert een punt binnen een vierhoek (quad).
    p00: Top-Left, p10: Top-Right, p01: Bottom-Left, p11: Bottom-Right
    u, v: parameters tussen 0 en 1 (0.5 is het midden)
    """
    # X interpolatie
    x_top = p00['x'] * (1 - u) + p10['x'] * u
    x_bot = p01['x'] * (1 - u) + p11['x'] * u
    final_x = x_top * (1 - v) + x_bot * v

    # Y interpolatie
    y_top = p00['y'] * (1 - u) + p10['y'] * u
    y_bot = p01['y'] * (1 - u) + p11['y'] * u
    final_y = y_top * (1 - v) + y_bot * v

    return final_x, final_y

def generate_full_dataset(num_samples):
    valid_shifts = get_valid_shifts(DIVISIONS, EDGE_LENGTH)

    all_vertices = []

    # Aantal vertices in Top Layer (5x5 = 25)
    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    for sample_id in range(num_samples):
        # --- STAP 1: TOP LAYER ---
        top_layer_coords = {} # Opslag voor snelle toegang bij stap 2
        vertex_idx = 0

        for i in range(num_nodes_y_top):
            for j in range(num_nodes_x_top):
                # Basis positie
                base_x = j * EDGE_LENGTH
                base_y = i * EDGE_LENGTH
                base_z = 0.0

                # Constraints checken
                is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
                is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
                is_corner = is_x_edge and is_y_edge

                shift_x = 0.0
                shift_y = 0.0
                v_type = "Top_Inner"

                if is_corner:
                    v_type = "Top_Corner"
                    # Geen verplaatsing
                elif is_x_edge:
                    v_type = "Top_Edge"
                    # Verticale rand: beweegt alleen in Y
                    shift_y = random.choice(valid_shifts)
                elif is_y_edge:
                    v_type = "Top_Edge"
                    # Horizontale rand: beweegt alleen in X
                    shift_x = random.choice(valid_shifts)
                else:
                    # Inner: beweegt in beide
                    shift_x = random.choice(valid_shifts)
                    shift_y = random.choice(valid_shifts)

                # Save
                final_x = base_x + shift_x
                final_y = base_y + shift_y
                final_z = base_z

                # Opslaan in dictionary voor lookup (gebruik (row, col) als key)
                top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z, 'idx': vertex_idx}

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": vertex_idx,
                    "layer": "top",
                    "x": round(final_x, 3),
                    "y": round(final_y, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

        # --- STAP 2: LOWER LAYER ---
        # Lower layer heeft 1 vertex per cel (4x4 = 16 vertices)
        # We itereren over de CELLEN van de top layer

        for i in range(GRID_CELLS_Y):      # 0 tot 3
            for j in range(GRID_CELLS_X):  # 0 tot 3

                # Haal de 4 hoekpunten van de cel op uit de Top Layer
                # (Let op: i is rij (Y), j is kolom (X))
                # Grid oriëntatie: (0,0) is linksonder
                p_bl = top_layer_coords[(i, j)]         # Bottom-Left
                p_br = top_layer_coords[(i, j+1)]       # Bottom-Right
                p_tl = top_layer_coords[(i+1, j)]       # Top-Left
                p_tr = top_layer_coords[(i+1, j+1)]     # Top-Right

                # Bepaal positie binnen de cel
                # "Scaled down by 50%" betekent dat u en v tussen 0.25 en 0.75 liggen
                u = random.uniform(0.25, 0.75) # Positie tussen links en rechts
                v = random.uniform(0.25, 0.75) # Positie tussen onder en boven

                # Let op de volgorde voor interpolatie functie:
                # bilinear_interpolate(TopLeft, TopRight, BotLeft, BotRight, u, v)
                # Hier is u de fractie van Links naar Rechts (X-richting)
                # Hier is v de fractie van Boven naar Beneden? Of Onder naar Boven?
                # In mijn functie: v=0 is Top, v=1 is Bottom.
                # Dus: p00=TL, p10=TR, p01=BL, p11=BR

                # Even aanpassen aan mijn functie definitie:
                # p00 (Top-Left), p10 (Top-Right), p01 (Bottom-Left), p11 (Bottom-Right)
                lx, ly = bilinear_interpolate(p_tl, p_tr, p_bl, p_br, u, 1-v)
                # Note: 1-v omdat mijn functie v=0 als top ziet, maar i loopt op, dus i+1 is 'hoger' in Y.
                # Als Y omhoog gaat (standaard grafiek), is i+1 (hogere Y) de "Top".

                # Z-positie
                base_z = -LAYER_HEIGHT
                z_shift = random.choice(valid_shifts) # Dezelfde discrete stappen
                final_z = base_z + z_shift

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": vertex_idx,
                    "layer": "bottom",
                    "x": round(lx, 3),
                    "y": round(ly, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

    return pd.DataFrame(all_vertices)

# --- EDGES GENEREREN (Static Topology) ---
def generate_edges_topology():
    edges = []
    edge_id = 0

    # 1. Top Layer Edges (Grid 5x5)
    # Rijen
    for i in range(5):
        for j in range(4):
            u = i * 5 + j
            v = u + 1
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "top_horizontal"})
            edge_id += 1
    # Kolommen
    for i in range(4):
        for j in range(5):
            u = i * 5 + j
            v = u + 5
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "top_vertical"})
            edge_id += 1

    # 2. Lower Layer Edges (Grid 4x4)
    # Indices beginnen bij 25.
    start_idx = 25
    # Rijen (4 rijen van 3 segmenten? Nee, 4x4 punten heeft 4 rijen van 3 segmenten)
    # Wacht, Lower layer is 4x4 punten (16 punten). Dat is een 3x3 grid van cellen in de onderlaag?
    # Nee, er is 1 punt per toplaag-cel. Toplaag heeft 4x4 cellen. Dus onderlaag is 4x4 PUNTEN.
    # Dus onderlaag vormt een 3x3 grid van cellen.

    for i in range(4): # 4 rijen
        for j in range(3): # 3 verbindingen
            u = start_idx + i * 4 + j
            v = u + 1
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "bottom_horizontal"})
            edge_id += 1

    for i in range(3): # 3 rijen verbindingen
        for j in range(4): # 4 kolommen
            u = start_idx + i * 4 + j
            v = u + 4
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "bottom_vertical"})
            edge_id += 1

    return pd.DataFrame(edges)

# --- UITVOEREN ---
df_vertices_v2 = generate_full_dataset(NUM_SAMPLES)
df_edges_v2 = generate_edges_topology()

# Save
df_vertices_v2.to_csv("dataset_vertices_v2.csv", index=False)
df_edges_v2.to_csv("static_edges_v2.csv", index=False)

print("Vertices V2 (Top + Bottom):")
print(df_vertices_v2.groupby('layer').head(2)) # Show sample of both
print("\nEdges V2 (Topology):")
print(df_edges_v2.tail(5)) # Show last few edges (bottom layer)

Vertices V2 (Top + Bottom):
    sample_id  vertex_index   layer        x       y      z
0           0             0     top    0.000   0.000    0.0
1           0             1     top   62.500   0.000    0.0
25          0            25  bottom   62.583  64.787 -112.5
26          0            26  bottom  141.305  48.700 -137.5

Edges V2 (Topology):
    edge_id   u   v             type
59       59  32  36  bottom_vertical
60       60  33  37  bottom_vertical
61       61  34  38  bottom_vertical
62       62  35  39  bottom_vertical
63       63  36  40  bottom_vertical


## Full geometry generation

In [7]:
import pandas as pd
import numpy as np
import random

# --- CONFIGURATIE ---
GRID_CELLS_X = 2       # Aantal cellen in X
GRID_CELLS_Y = 2       # Aantal cellen in Y
EDGE_LENGTH = 300.0    # Afmeting van een cel
LAYER_HEIGHT = 150.0   # Afstand tussen top en bottom layer
DIVISIONS = 8          # Aantal stappen voor de discrete verplaatsing
NUM_SAMPLES = 5        # Aantal samples

GRID = f"{GRID_CELLS_X}x{GRID_CELLS_Y}"

In [19]:
def get_valid_shifts(divisions, edge_length):
    """Berekent de toegestane verschuivingen (verwijdert extremen)."""
    half_div = divisions // 2
    all_steps = list(range(-half_div, half_div + 1))
    valid_steps = all_steps[1:-1] # Verwijder eerste en laatste
    valid_shifts = [(step / divisions) * edge_length for step in valid_steps]
    return valid_shifts

def bilinear_interpolate(p00, p10, p01, p11, u, v):
    """
    Interpoleert een punt binnen een vierhoek.
    p00: Bottom-Left, p10: Bottom-Right, p01: Top-Left, p11: Top-Right (in standaard cartesiaans)
    Maar in matrix indexering (rij i, kol j):
    (i, j) is vaak Top-Left in images, maar Bottom-Left in Grasshopper/Cartesiaans als y omhoog gaat.
    Laten we aannemen: i=0 is y=0 (onder), i=max is y=max (boven).
    Dan is rij i "onder" rij i+1.
    p_bl = (i, j), p_br = (i, j+1)
    p_tl = (i+1, j), p_tr = (i+1, j+1)
    """
    # X interpolatie
    # Onderkant (row i)
    x_bot = p00['x'] * (1 - u) + p10['x'] * u
    # Bovenkant (row i+1)
    x_top = p01['x'] * (1 - u) + p11['x'] * u

    final_x = x_bot * (1 - v) + x_top * v

    # Y interpolatie
    y_bot = p00['y'] * (1 - u) + p10['y'] * u
    y_top = p01['y'] * (1 - u) + p11['y'] * u

    final_y = y_bot * (1 - v) + y_top * v

    return final_x, final_y

def generate_full_dataset(num_samples):
    valid_shifts = get_valid_shifts(DIVISIONS, EDGE_LENGTH)
    all_vertices = []

    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    for sample_id in range(num_samples):
        # --- STAP 1: TOP LAYER ---
        top_layer_coords = {}
        vertex_idx = 0

        for i in range(num_nodes_y_top): # i is de rij index (y-richting)
            for j in range(num_nodes_x_top): # j is de kolom index (x-richting)
                # Basis positie
                base_x = j * EDGE_LENGTH
                base_y = i * EDGE_LENGTH
                base_z = 0.0

                # Constraints
                is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
                is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
                is_corner = is_x_edge and is_y_edge

                shift_x = 0.0
                shift_y = 0.0

                if is_corner:
                    pass # Corner: Vast
                elif is_x_edge:
                    # Verticale rand (links/rechts): beweegt in Y
                    shift_y = random.choice(valid_shifts)
                elif is_y_edge:
                    # Horizontale rand (onder/boven): beweegt in X
                    shift_x = random.choice(valid_shifts)
                else:
                    # Inner: beweegt in X en Y
                    shift_x = random.choice(valid_shifts)
                    shift_y = random.choice(valid_shifts)

                final_x = base_x + shift_x
                final_y = base_y + shift_y
                final_z = base_z

                top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z}

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "top",
                    "x": round(final_x, 3),
                    "y": round(final_y, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

        # --- STAP 2: LOWER LAYER ---
        # Vertex index loopt door vanaf 25
        # i loop 0..3, j loop 0..3

        for i in range(GRID_CELLS_Y):
            for j in range(GRID_CELLS_X):
                # We pakken de 4 hoekpunten van de top-cel
                # Rij i is 'onder', Rij i+1 is 'boven'
                p_bl = top_layer_coords[(i, j)]       # Bottom-Left
                p_br = top_layer_coords[(i, j+1)]     # Bottom-Right
                p_tl = top_layer_coords[(i+1, j)]     # Top-Left
                p_tr = top_layer_coords[(i+1, j+1)]   # Top-Right

                # Random positie in de cel (scaled 50%)
                u = random.uniform(0.25, 0.75)
                v = random.uniform(0.25, 0.75)

                # Interpoleren (let op volgorde parameters van mijn functie)
                # def bilinear_interpolate(p00, p10, p01, p11, u, v):
                # p00=BL, p10=BR, p01=TL, p11=TR
                lx, ly = bilinear_interpolate(p_bl, p_br, p_tl, p_tr, u, v)

                # Z positie
                base_z = -LAYER_HEIGHT
                z_shift = random.choice(valid_shifts)
                final_z = base_z + z_shift

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "bottom",
                    "x": round(lx, 3),
                    "y": round(ly, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

    return pd.DataFrame(all_vertices)

In [23]:
def generate_dynamic_edges_per_sample(num_samples, cells_x, cells_y):
    edges_data = []

    # Bereken hulpparameters
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top_vertices = nodes_x_top * nodes_y_top

    # We itereren door elke sample om de edges per sample vast te leggen
    for sample_id in range(num_samples):

        edge_counter = 0  # Reset edge counter per sample (of wil je unieke ID's over de hele file? Meestal per sample resetten: e0..e127)

        # Hulpfunctie om edge toe te voegen
        def add_edge(u, v):
            nonlocal edge_counter
            edges_data.append({
                "sample_id": sample_id,
                "edge": f"e{edge_counter}",
                "V1": u,
                "V2": v,
            })
            edge_counter += 1

        # --- 1. TOP LAYER GRID ---
        # Vertices 0 tot num_top_vertices-1
        for r in range(nodes_y_top):      # loop rijen
            for c in range(nodes_x_top):  # loop kolommen
                current = r * nodes_x_top + c

                # Horizontaal (naar rechts)
                if c < cells_x: # zolang niet de laatste kolom
                    add_edge(current, current + 1)

                # Verticaal (naar beneden, of 'boven' in matrix index)
                if r < cells_y: # zolang niet de laatste rij
                    add_edge(current, current + nodes_x_top)

        # --- 2. BOTTOM LAYER GRID ---
        # Start index is na de laatste top vertex
        start_idx_bottom = num_top_vertices

        # Bottom grid heeft evenveel punten als er cellen zijn (cells_x * cells_y)
        # Maar de grid verbindingen zijn er eentje minder dan het aantal punten
        # Bottom punten zijn een grid van (cells_x) breed bij (cells_y) hoog.

        for r in range(cells_y):
            for c in range(cells_x):
                current = start_idx_bottom + r * cells_x + c

                # Horizontaal (naar rechts)
                if c < cells_x - 1:
                    add_edge(current, current + 1)

                # Verticaal (naar beneden)
                if r < cells_y - 1:
                    add_edge(current, current + cells_x)

        # --- 3. DIAGONALS (Pyramid connections) ---
        # Verbind elke Bottom vertex met de 4 Top vertices erboven
        for r in range(cells_y):
            for c in range(cells_x):
                bottom_node = start_idx_bottom + r * cells_x + c

                # De 4 corresponderende punten in de Top layer
                # Top grid is (cells_x + 1) breed
                top_tl = r * nodes_x_top + c               # Top-Left (of row i)
                top_tr = r * nodes_x_top + (c + 1)         # Top-Right
                top_bl = (r + 1) * nodes_x_top + c         # Bottom-Left (row i+1)
                top_br = (r + 1) * nodes_x_top + (c + 1)   # Bottom-Right

                add_edge(bottom_node, top_tl)
                add_edge(bottom_node, top_tr)
                add_edge(bottom_node, top_bl)
                add_edge(bottom_node, top_br)

    return pd.DataFrame(edges_data)

# --- UITVOEREN ---
df_final_edges = generate_dynamic_edges_per_sample(NUM_SAMPLES, GRID_CELLS_X, GRID_CELLS_Y)

# Opslaan
csv_filename = "dataset_edges_full_dynamic.csv"
df_final_edges.to_csv(csv_filename, index=False)

print(f"Generatie gereed voor {NUM_SAMPLES} samples.")
print(f"Grid grootte: {GRID_CELLS_X}x{GRID_CELLS_Y}")
print(f"Totaal aantal regels in CSV: {len(df_final_edges)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(df_final_edges.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(df_final_edges[df_final_edges['sample_id'] == 1].head(5))

Generatie gereed voor 5 samples.
Grid grootte: 2x2
Totaal aantal regels in CSV: 160

Voorbeeld output (eerste 5 regels van sample 0):
   sample_id edge  V1  V2
0          0   e0   0   1
1          0   e1   0   3
2          0   e2   1   2
3          0   e3   1   4
4          0   e4   2   5

Voorbeeld output (eerste 5 regels van sample 1):
    sample_id edge  V1  V2
32          1   e0   0   1
33          1   e1   0   3
34          1   e2   1   2
35          1   e3   1   4
36          1   e4   2   5


In [20]:
# --- EXECUTE ---
final_vertices = generate_full_dataset(NUM_SAMPLES)
final_edges = generate_clean_edge_list()

# Save
final_vertices.to_csv(f"dataset_vertices_{GRID}_{NUM_SAMPLES}.csv", index=False)
final_edges.to_csv(f"dataset_edges_{GRID}_{NUM_SAMPLES}.csv", index=False)

print("Final Generation Complete.")
print("Vertices Shape:", final_vertices.shape)
print("Edges Shape:", final_edges.shape)
print("\nSample Vertices:")
print(final_vertices.head())
print("\nSample Edges (Diagonals):")
print(final_edges[final_edges['Edge'] == 'diagonal'].head())

Final Generation Complete.
Vertices Shape: (65, 6)
Edges Shape: (128, 2)

Sample Vertices:
   sample_id vertex_index layer      x      y    z
0          0           v0   top    0.0    0.0  0.0
1          0           v1   top  300.0    0.0  0.0
2          0           v2   top  600.0    0.0  0.0
3          0           v3   top    0.0  187.5  0.0
4          0           v4   top  300.0  300.0  0.0

Sample Edges (Diagonals):
Empty DataFrame
Columns: [Edge, Vertices]
Index: []


## Graveyard

In [ ]:
# @title
def generate_static_topology():
    edges = []
    edge_id = 0

    num_nodes_x_top = GRID_CELLS_X + 1 # 5

    # --- 1. Top Layer Grid Edges ---
    # Horizontaal (Rijen)
    for i in range(5):
        for j in range(4):
            u = i * 5 + j
            v = u + 1
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "top_horizontal"})
            edge_id += 1
    # Verticaal (Kolommen)
    for i in range(4):
        for j in range(5):
            u = i * 5 + j
            v = u + 5
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "top_vertical"})
            edge_id += 1

    # --- 2. Lower Layer Grid Edges (optioneel in sommige spaceframes, maar fig 3.1.2 toont een grid) ---
    # Bottom vertices start index = 25
    # Grid is 4x4
    start_bottom = 25

    # Horizontaal
    for i in range(4):
        for j in range(3):
            u = start_bottom + i * 4 + j
            v = u + 1
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "bottom_horizontal"})
            edge_id += 1
    # Verticaal
    for i in range(3):
        for j in range(4):
            u = start_bottom + i * 4 + j
            v = u + 4
            edges.append({"edge_id": edge_id, "u": u, "v": v, "type": "bottom_vertical"})
            edge_id += 1

    # --- 3. In-between (Diagonal / Pyramid) Edges ---
    # Verbindt elke Bottom vertex met de 4 omliggende Top vertices

    for i in range(GRID_CELLS_Y):      # 0..3
        for j in range(GRID_CELLS_X):  # 0..3

            # Index van de huidige Bottom Vertex
            bottom_v = start_bottom + i * 4 + j

            # Indices van de 4 Top Vertices rondom deze cel
            # Rij i en Rij i+1
            top_bl = i * 5 + j           # Bottom-Left
            top_br = i * 5 + (j + 1)     # Bottom-Right
            top_tl = (i + 1) * 5 + j     # Top-Left
            top_tr = (i + 1) * 5 + (j + 1) # Top-Right

            # Maak 4 verbindingen
            edges.append({"edge_id": edge_id, "u": bottom_v, "v": top_bl, "type": "diagonal"})
            edge_id += 1
            edges.append({"edge_id": edge_id, "u": bottom_v, "v": top_br, "type": "diagonal"})
            edge_id += 1
            edges.append({"edge_id": edge_id, "u": bottom_v, "v": top_tl, "type": "diagonal"})
            edge_id += 1
            edges.append({"edge_id": edge_id, "u": bottom_v, "v": top_tr, "type": "diagonal"})
            edge_id += 1